# Multi-Agent Systems with LangGraph
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/19-multi-agent-notebook/multi_agent_workbook.ipynb)

**Multi-agent systems** coordinate multiple specialized LLM agents to tackle complex tasks that a single agent handles poorly — either because the task is too broad, requires different tools, or benefits from specialization and cross-checking. By the end of this session you will understand *why* multi-agent architectures exist, *how* the supervisor/worker pattern works, and *how* to build, stream, and debug a production-ready multi-agent graph with LangGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why multi-agent? Single vs multi-agent trade-offs |
| 2 | **Shared State** — `AgentState`, `add_messages` reducer, message passing |
| 3 | **Worker Agents** — `create_react_agent`, tools, system prompts |
| 4 | **Supervisor Node** — `with_structured_output`, typed routing decisions |
| 5 | **Graph Assembly** — `StateGraph`, `add_conditional_edges`, the routing loop |
| 6 | **Streaming & Debugging** — `stream(stream_mode='updates')`, prompt inspection |
| 7 | **Failure Modes** — loops, stalls, context overflow, missing tool calls |
| ★ | **Advanced Patterns** — fan-out with `Send`, checkpointing, parallel workers |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442
>
> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155
>
> LangGraph multi-agent concepts: https://langchain-ai.github.io/langgraph/concepts/multi_agent/
>
> LangGraph `create_react_agent` reference: https://langchain-ai.github.io/langgraph/reference/prebuilt/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain==0.3.27",
            "langchain-core==0.3.79",
            "langchain-openai==0.3.33",
            "langchain-community==0.3.31",
            "langgraph==0.6.7",
            "langgraph-prebuilt==0.6.4",
            "duckduckgo-search==8.1.1",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not set or invalid. "
    "Local: add it to .env. Colab: use the Secrets panel."
)
print(f"OPENAI_API_KEY present: True ({key[:8]}...)")

---

## Part 1 — Why Multi-Agent? Concepts and Trade-offs

---

### The Problem with Single Agents

A single LLM agent with many tools quickly runs into limits:

1. **Context overflow** — the agent's entire reasoning trace is in one context window. Long tasks exhaust the limit.
2. **Tool confusion** — give one agent 20 tools and it starts calling the wrong one.
3. **Quality** — a generalist agent produces mediocre output on every sub-task; a specialist excels at one.
4. **No cross-checking** — the agent cannot critique its own output without external structure.

Multi-agent systems solve these by distributing the work.

---

### Single Agent vs Multi-Agent

| Dimension | Single Agent | Multi-Agent |
|-----------|-------------|-------------|
| **Context window** | One shared window; overflows on long tasks | Each agent has its own context; coordinator sees summaries |
| **Tool count** | All tools in one agent (confusion risk) | Each agent has focused, minimal toolset |
| **Quality** | Generalist output | Specialist output per sub-task |
| **Parallelism** | Sequential only | Workers can run in parallel (`Send` API) |
| **Debugging** | Hard — single opaque trace | Easy — inspect each node's output independently |
| **Cost** | One model call per step | Multiple calls; optimize by routing cheap tasks to smaller models |
| **Failure isolation** | One error crashes the whole task | Individual worker failures are catchable |

---

### Orchestration Patterns

| Pattern | Structure | When to use |
|---------|-----------|-------------|
| **Supervisor/Worker** | Coordinator routes tasks to specialists | Most production tasks — clear sub-task boundaries |
| **Pipeline** | Agent A → Agent B → Agent C (linear) | When steps have a fixed, known order |
| **Hierarchical** | Supervisor of supervisors | Very large systems with nested domains |
| **Peer-to-peer** | Agents message each other freely | Debate, red-team, adversarial review |
| **Fan-out / Fan-in** | Parallel workers → merge node | Independent sub-tasks that can run concurrently |

This workshop implements **Supervisor/Worker** — the most common production pattern.

---

### System Topology

```
┌─────────────────────────────────────────────────────────────────┐
│                      MULTI-AGENT SYSTEM                         │
│                                                                 │
│   User query                                                    │
│       │                                                         │
│       ▼                                                         │
│  ┌──────────────┐   next=researcher   ┌─────────────────┐       │
│  │              │ ──────────────────► │ researcher_agent│       │
│  │  SUPERVISOR  │                     │  (web search)   │       │
│  │  (router)    │◄────────────────────│                 │       │
│  │              │   appends message   └─────────────────┘       │
│  │              │                                               │
│  │              │   next=writer       ┌─────────────────┐       │
│  │              │ ──────────────────► │  writer_agent   │       │
│  │              │                     │  (synthesis)    │       │
│  │              │◄────────────────────│                 │       │
│  │              │   appends message   └─────────────────┘       │
│  │              │                                               │
│  │              │   next=FINISH                                 │
│  └──────────────┘ ────────────────────────────────────► END     │
│                                                                 │
│   Shared: AgentState { messages: [...], next: str }             │
└─────────────────────────────────────────────────────────────────┘
```

**Key insight**: all agents share a single `AgentState`. The `add_messages` reducer *appends* new messages rather than overwriting, so the supervisor always sees the full conversation when deciding what to do next.

---

### Communication Strategies

| Strategy | How messages flow | LangGraph mechanism |
|----------|-------------------|---------------------|
| **Shared message list** | All agents read/write one list | `add_messages` reducer on `AgentState` |
| **State fields** | Typed fields passed between nodes | Additional fields on `TypedDict` |
| **Direct send** | Parent sends specific input to child | `Send` API (fan-out) |
| **Tool call** | Agent invokes another agent as a tool | Subgraph or custom tool wrapping |

This workshop uses **shared message list** — the simplest and most common approach.

---

## Part 2 — Shared State with `add_messages`

---

### What Is `AgentState`?

Every LangGraph graph has a **state schema** — a `TypedDict` that defines what data flows through the graph. Each node reads from state and returns a partial update.

The **reducer** on a field controls how updates are merged:

| Reducer | Behavior | Use case |
|---------|----------|----------|
| *(none / default)* | Last write wins — new value replaces old | Simple scalars like `next` |
| `add_messages` | Appends new messages to the list | Shared conversation history |

`add_messages` is what makes multi-agent message passing work: each worker appends its response to the shared list without knowing about or overwriting what other agents wrote.

### Message Flow Through State

```
Initial:         messages=[HumanMessage("query")],            next=""
                               │
                        supervisor_node
                               │  returns {next: "researcher"}
                               ▼
After step 1:    messages=[HumanMessage("query")],            next="researcher"
                               │
                        researcher_node
                               │  returns {messages: [AIMessage("[researcher] ...")]}
                               ▼
After step 2:    messages=[HumanMessage, AIMessage(researcher)], next="researcher"
                               │
                        supervisor_node (sees FULL history)
                               │  returns {next: "writer"}
                               ▼
After step 3:    messages=[HumanMessage, AIMessage(researcher)], next="writer"
                               ...
```

In [ ]:
# ─── 2-A: Define AgentState ───────────────────────────────────────────────────
# The TypedDict that every node reads from and writes to.

from typing import Annotated

from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # reducer: appends, never overwrites
    next: str  # supervisor routing decision; default reducer overwrites


print("AgentState defined.")
print("  messages: add_messages reducer — new messages are APPENDED to the list")
print("  next:     last-write-wins — supervisor overwrites this each turn")

In [ ]:
# ─── 2-B: Demonstrate the add_messages reducer ────────────────────────────────
# Understand exactly what 'appends' means before wiring the graph.

from langchain_core.messages import AIMessage, HumanMessage

# Simulate what add_messages does when a node returns new messages
current_messages = [HumanMessage(content="What is LangGraph?")]

# Node A returns one message — the reducer APPENDS, does not replace
update_from_node_a = [
    AIMessage(
        content="[researcher] LangGraph is a graph execution framework.",
        name="researcher",
    )
]

merged = current_messages + update_from_node_a  # this is what the reducer does internally

print(f"Before:       {len(current_messages)} message(s)")
print(f"Node returned: {len(update_from_node_a)} message(s)")
print(f"After merge:  {len(merged)} message(s)")
print()
for i, msg in enumerate(merged):
    role = getattr(msg, "name", None) or msg.type.upper()
    print(f"  [{i}] {role}: {msg.content[:80]}")

---

## Part 3 — Worker Agents with `create_react_agent`

---

### What Is a ReAct Agent?

**ReAct** (Reason + Act) is the standard single-agent loop:

```
┌──────────────────────────────────────────────────┐
│                 ReAct Loop                       │
│                                                  │
│   Input messages                                 │
│       │                                          │
│       ▼                                          │
│  ┌─────────┐  tool_call?  ┌──────────────┐       │
│  │   LLM   │ ───────────► │  Tool Runner │       │
│  └────┬────┘              └──────┬───────┘       │
│       │◄──── tool result ────────┘               │
│       │                                          │
│       │ no tool_call (final answer)              │
│       ▼                                          │
│  Return messages                                 │
└──────────────────────────────────────────────────┘
```

`create_react_agent` from `langgraph.prebuilt` builds this entire loop in one call — you supply an LLM, a list of tools, and an optional system prompt.

### Worker Design Decisions

| Decision | Our choice | Rationale |
|----------|-----------|----------|
| LLM for workers | `gpt-4o-mini` | Cost-effective for leaf nodes; reserve larger models for the supervisor |
| researcher tools | `DuckDuckGoSearchResults` | Free, no API key, good for demos |
| writer tools | none | Writer synthesizes — giving it tools causes distraction |
| System prompt | `SystemMessage` | Injected before every invocation; defines the worker's specialty |

In [ ]:
# ─── 3-A: Initialize LLM and tools ───────────────────────────────────────────

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

# Shared LLM — use gpt-4o-mini for workers to keep costs low
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# The researcher's only tool: web search
search_tool = DuckDuckGoSearchResults(max_results=3)

print("LLM: gpt-4o-mini (temperature=0 for deterministic routing)")
print(f"Search tool: {search_tool.name} — returns up to 3 results")
print()
print(f"Tool description: {search_tool.description[:120]}...")

In [ ]:
# ─── 3-B: Build the worker agents ────────────────────────────────────────────
# create_react_agent wraps LLM + tools into a complete ReAct subgraph.

from langgraph.prebuilt import create_react_agent

researcher_agent = create_react_agent(
    llm,
    tools=[search_tool],
    prompt=SystemMessage(
        content=(
            "You are a research specialist. Search the web to find accurate, "
            "current information. Return a factual summary with key points. "
            "Always note where you found each fact."
        )
    ),
)

writer_agent = create_react_agent(
    llm,
    tools=[],  # writer does not search — it synthesizes only
    prompt=SystemMessage(
        content=(
            "You are a writing specialist. Synthesize the research findings "
            "into a clear, well-structured, concise answer. "
            "Use bullet points for lists and bold for key terms."
        )
    ),
)

print("Worker agents ready:")
print("  researcher_agent: gpt-4o-mini + DuckDuckGo search")
print("  writer_agent:     gpt-4o-mini + no tools (synthesis only)")

In [ ]:
# ─── 3-C: Test a worker agent in isolation ────────────────────────────────────
# ALWAYS verify workers work standalone before wiring them into the graph.
# This is the most useful early debugging step.

test_query = "What is LangGraph and when was it first released?"

print(f"Testing researcher_agent in isolation...")
print(f"Query: '{test_query}'\n")

result = researcher_agent.invoke({"messages": [HumanMessage(content=test_query)]})

print(f"Messages returned: {len(result['messages'])}")
print()
for msg in result["messages"]:
    role = getattr(msg, "name", None) or msg.type.upper()
    content_preview = (
        msg.content[:200].replace("\n", " ") if msg.content else "[tool call — no text content]"
    )
    print(f"  [{role}] {content_preview}...")

---

## Part 4 — The Supervisor Node

---

### How the Supervisor Works

The supervisor is a single LLM call that:
1. Reads the full message history from `state['messages']`
2. Returns a **typed routing decision** — not free-form text
3. Writes `next` to state, which `add_conditional_edges` reads to pick the next node

The critical pattern is `with_structured_output(SupervisorDecision)`. This forces the LLM to return valid JSON matching a Pydantic schema — no parsing, no regex, no hallucinated node names.

```
messages history
      │
      ▼
┌──────────────────────────────────────────────┐
│  SUPERVISOR CHAIN                            │
│                                              │
│  SUPERVISOR_PROMPT (system + history)        │
│       │                                      │
│       ▼                                      │
│  llm.with_structured_output(SupervisorDecision) │
│       │                                      │
│       ▼                                      │
│  SupervisorDecision(next="researcher")       │
└──────────────────────────────────────────────┘
      │
      ▼
  state["next"] = "researcher"
```

### Why `Literal` Types Matter

```python
class SupervisorDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]
```

- **`Literal`** constrains the LLM to exactly three valid strings — any other output raises a Pydantic validation error immediately, rather than silently routing to a non-existent node.
- **Type safety at the boundary**: errors are caught before touching the graph, not during execution.

In [ ]:
# ─── 4-A: Define the routing schema and supervisor chain ──────────────────────

from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel


class SupervisorDecision(BaseModel):
    """Typed routing decision returned by the supervisor."""

    next: Literal["researcher", "writer", "FINISH"]


SUPERVISOR_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a supervisor coordinating two specialist agents:
- researcher: searches the web for current facts and information
- writer:     synthesizes research into a clear, structured answer

Rules:
1. Always call researcher first to gather facts.
2. Call writer after researcher has returned findings.
3. Reply FINISH once the writer has produced a final answer.
4. Never call the same agent twice in a row.""",
        ),
        ("human", "Conversation so far:\n{history}\n\nWho should act next?"),
    ]
)

# with_structured_output forces valid JSON matching SupervisorDecision schema
supervisor_chain = SUPERVISOR_PROMPT | llm.with_structured_output(SupervisorDecision)

print("Supervisor chain ready.")
print("  Input:  message history (formatted as string)")
print("  Output: SupervisorDecision(next='researcher' | 'writer' | 'FINISH')")

In [ ]:
# ─── 4-B: Test the supervisor in isolation ────────────────────────────────────
# Verify routing logic before wiring into the graph.

# Scenario 1: empty history — supervisor should route to researcher first
decision_1 = supervisor_chain.invoke(
    {"history": "HUMAN: What are the main LangGraph features?"}
)
print(f"Scenario 1 (no research yet):    next='{decision_1.next}'")
assert decision_1.next == "researcher", "Expected researcher first"

# Scenario 2: research done, no writing yet — should route to writer
decision_2 = supervisor_chain.invoke(
    {
        "history": (
            "HUMAN: What are the main LangGraph features?\n"
            "AI: [researcher] LangGraph provides stateful graph execution, "
            "conditional edges, and streaming."
        )
    }
)
print(f"Scenario 2 (research done):      next='{decision_2.next}'")
assert decision_2.next == "writer", "Expected writer after research"

# Scenario 3: both done — should FINISH
decision_3 = supervisor_chain.invoke(
    {
        "history": (
            "HUMAN: What are the main LangGraph features?\n"
            "AI: [researcher] LangGraph provides stateful graph execution, "
            "conditional edges, and streaming.\n"
            "AI: [writer] **LangGraph** is a framework for building stateful "
            "multi-step AI applications. Key features include stateful graphs, "
            "conditional routing, and real-time streaming."
        )
    }
)
print(f"Scenario 3 (both done):          next='{decision_3.next}'")
assert decision_3.next == "FINISH", "Expected FINISH after writer"

print()
print("All supervisor routing scenarios passed.")

---

## Part 5 — Assembling the Graph

---

### Graph Structure

```
START
  │
  ▼
supervisor ──── next=researcher ──► researcher ──┐
  ▲                                             │
  └─────────────────────────────────────────────┘
  │
supervisor ──── next=writer    ──► writer     ──┐
  ▲                                             │
  └─────────────────────────────────────────────┘
  │
supervisor ──── next=FINISH   ──► END
```

### Key Graph Concepts

| Concept | What it does |
|---------|-------------|
| `StateGraph(AgentState)` | Creates the graph with our state schema |
| `add_node(name, fn)` | Registers a node — `fn(state) -> dict` |
| `add_edge(a, b)` | Unconditional edge — always transitions from a to b |
| `add_conditional_edges(src, fn, map)` | Calls `fn(state)` to pick the next node |
| `graph.compile()` | Compiles into an executable `app` |

The **routing function** is pure Python — it reads `state['next']` and returns the node name. `add_conditional_edges` maps return values to actual node names.

In [ ]:
# ─── 5-A: Define node functions ──────────────────────────────────────────────
# Each node is a function: AgentState -> dict (partial state update).


def supervisor_node(state: AgentState) -> dict:
    """Read full message history, ask supervisor chain who acts next."""
    history = "\n".join(
        f"{(getattr(m, 'name', None) or m.type).upper()}: {m.content[:300]}"
        for m in state["messages"]
    )
    decision = supervisor_chain.invoke({"history": history})
    return {"next": decision.next}  # only updates the `next` field


def researcher_node(state: AgentState) -> dict:
    """Invoke researcher agent and append its final message to shared state."""
    result = researcher_agent.invoke({"messages": state["messages"]})
    last = result["messages"][-1]
    # Tag the message so the supervisor can identify its source in history
    tagged = AIMessage(content=f"[researcher] {last.content}", name="researcher")
    return {"messages": [tagged]}  # add_messages reducer appends this


def writer_node(state: AgentState) -> dict:
    """Invoke writer agent and append its final message to shared state."""
    result = writer_agent.invoke({"messages": state["messages"]})
    last = result["messages"][-1]
    tagged = AIMessage(content=f"[writer] {last.content}", name="writer")
    return {"messages": [tagged]}


def route(state: AgentState) -> Literal["researcher", "writer", "__end__"]:
    """Pure routing function: read state['next'] and return the node to visit."""
    nxt = state.get("next", "FINISH")
    return "__end__" if nxt == "FINISH" else nxt


print("Node functions defined:")
print("  supervisor_node  — reads history, returns {next: ...}")
print("  researcher_node  — invokes researcher agent, appends tagged AIMessage")
print("  writer_node      — invokes writer agent, appends tagged AIMessage")
print("  route            — maps state['next'] to node name")

In [ ]:
# ─── 5-B: Assemble and compile the graph ─────────────────────────────────────

from langgraph.graph import END, START, StateGraph

graph = StateGraph(AgentState)

# Register nodes
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)

# Entry point: always start at supervisor
graph.add_edge(START, "supervisor")

# Supervisor decides who goes next (conditional on state['next'])
graph.add_conditional_edges(
    "supervisor",
    route,
    {
        "researcher": "researcher",
        "writer": "writer",
        "__end__": END,
    },
)

# Workers always loop back to supervisor for the next routing decision
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")

app = graph.compile()

print("Graph compiled.")
print("Edges:")
print("  START       -> supervisor")
print("  supervisor  -> researcher | writer | END  (conditional on state['next'])")
print("  researcher  -> supervisor")
print("  writer      -> supervisor")

In [ ]:
# ─── 5-C: Visualise the graph ────────────────────────────────────────────────
# Requires graphviz / mermaid — works in Colab and most local Jupyter installs.

try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph render unavailable ({e}) — printing Mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

---

## Part 6 — Streaming and Debugging

---

### Two Execution Modes

| Method | Returns | When to use |
|--------|---------|-------------|
| `app.invoke(state)` | Final state dict | Production — you only need the answer |
| `app.stream(state, stream_mode='updates')` | Iterator of `{node_name: partial_state}` | Debugging — see every step as it happens |

`stream_mode='updates'` emits one dict per node as it completes. This is the primary debugging tool for multi-agent graphs — you can trace exactly which agent fired, what the supervisor decided, and what each worker returned, all in real time.

### The `MAX_TURNS` Guard

Always set a hard limit on streaming iterations. If the supervisor has a bug and never routes to FINISH, the loop will run forever without a guard.

```python
for i, step in enumerate(app.stream(...)):
    if i >= MAX_TURNS:
        print("[MAX_TURNS reached — terminating]")
        break
```

In [ ]:
# ─── 6-A: Stream step-by-step execution ──────────────────────────────────────
# See exactly what each node does as the graph executes.

TASK = (
    "What are the main differences between LangGraph and AutoGen "
    "for building multi-agent systems?"
)
MAX_TURNS = 8  # guard against infinite supervisor loops

print(f"Task: {TASK}\n")
print("--- Streaming execution ---\n")

init_state = {"messages": [HumanMessage(content=TASK)], "next": ""}

for i, step in enumerate(app.stream(init_state, stream_mode="updates")):
    if i >= MAX_TURNS:
        print("[MAX_TURNS reached — stopping]")
        break
    node_name = list(step.keys())[0]
    node_out = step[node_name]
    print(f"[{node_name}]")
    if "next" in node_out:
        print(f"  routing -> {node_out['next']}")
    if "messages" in node_out:
        for msg in node_out["messages"]:
            preview = msg.content[:200].replace("\n", " ")
            suffix = "..." if len(msg.content) > 200 else ""
            print(f"  msg: {preview}{suffix}")
    print()

In [ ]:
# ─── 6-B: Full invoke and inspect final state ─────────────────────────────────
# invoke() runs the graph to completion and returns the complete final state.

TASK2 = "Explain the Agent-to-Agent (A2A) protocol and write a 3-sentence summary of its use cases."

result = app.invoke(
    {"messages": [HumanMessage(content=TASK2)], "next": ""},
)

print("=== Full Message History ===")
for msg in result["messages"]:
    role = getattr(msg, "name", None) or msg.type.upper()
    print(f"\n[{role.upper()}]")
    print(msg.content)

print("\n=== Final Answer (last message) ===")
print(result["messages"][-1].content)

In [ ]:
# ─── 6-C: Inspect message history size and token cost estimate ────────────────
# Multi-agent systems accumulate long histories. Track this — it drives cost.
# The supervisor re-reads the ENTIRE history on every turn; this compounds.

print(f"Total messages in final state: {len(result['messages'])}")
print()

total_chars = 0
total_approx_tokens = 0
for i, msg in enumerate(result["messages"]):
    name = getattr(msg, "name", None) or msg.type
    chars = len(msg.content)
    approx_tokens = chars // 4
    total_chars += chars
    total_approx_tokens += approx_tokens
    print(f"  [{i}] {name:<15} {chars:>5} chars  (~{approx_tokens:>4} tokens)")

print(f"\nTotal characters:             {total_chars}")
print(f"Approx total tokens consumed: ~{total_approx_tokens}")
print()
print("Tip: trim earlier messages or summarize state to keep context costs low.")
print("     Each supervisor call re-reads the ENTIRE history — this compounds fast.")

---

## Part 7 — Failure Modes and Debugging Checklist

---

### Common Multi-Agent Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Infinite loop** | Graph never reaches END | Supervisor never routes FINISH | Add turn counter or `MAX_TURNS` guard |
| **Routing hallucination** | Supervisor picks non-existent node | Free-text routing without schema | Use `with_structured_output` + `Literal` |
| **Worker stall** | Worker calls tool repeatedly | Tool returns poor results | Limit `max_iterations` on `create_react_agent` |
| **Context overflow** | Degraded output on long tasks | History grows unbounded | Summarize messages; use `MemorySaver` with trimming |
| **Message overwrite** | Previous agent output disappears | Wrong reducer (no `add_messages`) | Always use `Annotated[list, add_messages]` |
| **Supervisor indecision** | Alternates between same two workers | Weak prompt ordering rules | Add explicit ordering rules to supervisor prompt |
| **Silent tool failure** | Worker claims it searched but didn't | Tool import error silently caught | Test each worker in isolation (cell 3-C) |

---

### Debugging Checklist

1. **Test each worker in isolation** before building the graph (see cell 3-C)
2. **Test the supervisor in isolation** with known inputs (see cell 4-B)
3. **Stream with `stream_mode='updates'`** to see every node's output
4. **Print state at each step** — especially `state['next']` and `len(state['messages'])`
5. **Add a `MAX_TURNS` guard** to every streaming loop
6. **Check message count** after the run — unexpectedly high = context leak

In [ ]:
# ─── 7-A: Turn counter guard — prevent infinite loops ────────────────────────
# Add a turn_count field to state. If it exceeds a limit,
# supervisor_node forces FINISH regardless of what the LLM decides.


class SafeAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    turn_count: int  # incremented by supervisor on every call


MAX_SUPERVISOR_TURNS = 5


def safe_supervisor_node(state: SafeAgentState) -> dict:
    turn = state.get("turn_count", 0) + 1
    if turn >= MAX_SUPERVISOR_TURNS:
        print(f"  [GUARD] Max turns ({MAX_SUPERVISOR_TURNS}) reached — forcing FINISH")
        return {"next": "FINISH", "turn_count": turn}
    history = "\n".join(
        f"{(getattr(m, 'name', None) or m.type).upper()}: {m.content[:300]}"
        for m in state["messages"]
    )
    decision = supervisor_chain.invoke({"history": history})
    print(f"  [supervisor turn {turn}] -> {decision.next}")
    return {"next": decision.next, "turn_count": turn}


print("SafeAgentState adds turn_count field.")
print(f"Max supervisor turns before forced FINISH: {MAX_SUPERVISOR_TURNS}")

In [ ]:
# ─── 7-B: Stress test — task that doesn't obviously need research ─────────────
# Good check of the supervisor's routing rules under an edge case.

short_task = "Write a haiku about multi-agent systems."

print(f"Task: {short_task}\n")
print("--- Streaming ---\n")

for i, step in enumerate(
    app.stream(
        {"messages": [HumanMessage(content=short_task)], "next": ""},
        stream_mode="updates",
    )
):
    if i >= 8:
        print("[MAX_TURNS reached]")
        break
    node_name = list(step.keys())[0]
    node_out = step[node_name]
    if "next" in node_out:
        print(f"[{node_name}] -> {node_out['next']}")
    if "messages" in node_out:
        for msg in node_out["messages"]:
            print(f"[{node_name}] msg: {msg.content[:150].replace(chr(10), ' ')}")

---

## Part 8 ★ — Advanced Patterns (Bonus)

---

### Fan-out with the `Send` API

The `Send` API lets a single node dispatch messages to **multiple parallel workers** in one step. Each worker receives its own input; results are merged back in a `merge_node`.

```python
from langgraph.types import Send

def fan_out(state):
    sub_queries = ["LangGraph release date", "LangGraph vs AutoGen"]
    return [Send("researcher", {"messages": [HumanMessage(content=q)]}) for q in sub_queries]
```

Both researcher calls execute concurrently — results come back as a list that a merge node assembles.

---

### Persistent Checkpointing

Add a `MemorySaver` checkpointer to enable:
- **Multi-turn conversations** — re-invoke the same thread with a follow-up question
- **Time-travel debugging** — replay any prior state
- **Human-in-the-loop** — interrupt the graph and let a human modify state before resuming

```python
from langgraph.checkpoint.memory import MemorySaver

app = graph.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "session-42"}}
# Turn 1
app.invoke({"messages": [HumanMessage(content="What is LangGraph?")], "next": ""}, config)
# Turn 2 — graph remembers Turn 1 automatically
app.invoke({"messages": [HumanMessage(content="How does it compare to AutoGen?")], "next": ""}, config)
```

---

### Hierarchical Supervisors

For very large systems, build a supervisor-of-supervisors:

```
TOP-LEVEL SUPERVISOR
  ├── RESEARCH SUPERVISOR  ->  [web-searcher, paper-retriever, fact-checker]
  └── WRITING SUPERVISOR   ->  [drafter, editor, formatter]
```

Each sub-supervisor is itself a compiled subgraph invoked as a node in the parent graph.

In [ ]:
# ─── 8-A: Persistent checkpointing — multi-turn conversation ─────────────────
# Compile with MemorySaver so the graph remembers previous turns.

from langgraph.checkpoint.memory import MemorySaver

checkpointed_app = graph.compile(checkpointer=MemorySaver())

# thread_id ties invocations together — same ID = same conversation
config = {"configurable": {"thread_id": "workshop-session-1"}}

print("Turn 1 — initial question")
result_1 = checkpointed_app.invoke(
    {"messages": [HumanMessage(content="What is LangGraph used for?")], "next": ""},
    config=config,
)
print(f"  Messages after turn 1: {len(result_1['messages'])}")
print(f"  Final answer: {result_1['messages'][-1].content[:150]}...")

print()
print("Turn 2 — follow-up (graph remembers turn 1 automatically)")
result_2 = checkpointed_app.invoke(
    {"messages": [HumanMessage(content="How does it handle state across turns?")], "next": ""},
    config=config,  # same thread_id — graph loads prior state from checkpointer
)
print(f"  Messages after turn 2: {len(result_2['messages'])}")
print(f"  Final answer: {result_2['messages'][-1].content[:150]}...")

---

## Exercises

---

### Exercise 1 — Add a Fact-Checker Worker

Add a third worker `checker` that receives the writer's output and verifies claims against the researcher's findings. Add `'checker'` to `SupervisorDecision`'s `Literal` type, write a `checker_node`, and update the supervisor prompt to route to checker after writer.

**Expected routing order:** `researcher -> writer -> checker -> FINISH`

### Exercise 2 — Turn Counter in State

Add a `turn_count: int` field to `AgentState` (initialized to 0 on first invoke). Increment it in `supervisor_node`. If `turn_count >= 5`, force `next = 'FINISH'` regardless of what the LLM decides. Verify it terminates even if you deliberately break the supervisor prompt.

### Exercise 3 — Parallel Research with `Send`

Replace the single researcher call with two parallel sub-queries using the `Send` API:

```python
from langgraph.types import Send

def fan_out(state):
    queries = ["LangGraph architecture overview", "LangGraph vs AutoGen comparison"]
    return [Send("researcher", {"messages": [HumanMessage(content=q)]}) for q in queries]
```

Add a `merge_node` that combines both researcher results into a single message before passing to writer.

### Exercise 4 — Swap the Worker Model

Replace `gpt-4o-mini` in `researcher_agent` with `gpt-4o`. Run the same task and compare output quality, message length, and response time. What changes? Which tasks justify the higher cost?

In [ ]:
# ── Exercise 1 Starter — Add a fact-checker worker ───────────────────────────

# Step 1: extend the routing schema with 'checker'
class SupervisorDecisionV2(BaseModel):
    next: Literal["researcher", "writer", "checker", "FINISH"]


# Step 2: create the checker agent
checker_agent = create_react_agent(
    llm,
    tools=[],
    prompt=SystemMessage(
        content=(
            "You are a fact-checking specialist. Compare the writer's answer "
            "against the researcher's findings. Flag any unsupported claims. "
            "If everything checks out, confirm with a brief verification note."
        )
    ),
)


def checker_node(state: AgentState) -> dict:
    result = checker_agent.invoke({"messages": state["messages"]})
    last = result["messages"][-1]
    tagged = AIMessage(content=f"[checker] {last.content}", name="checker")
    return {"messages": [tagged]}


# TODO: update SUPERVISOR_PROMPT to include checker in the routing rules
# TODO: update supervisor_chain to use SupervisorDecisionV2
# TODO: add checker_node to the graph: graph.add_node("checker", checker_node)
# TODO: add edges: writer -> supervisor, checker -> supervisor
# TODO: update route() to handle 'checker' as a valid destination
print("Starter: checker_agent and checker_node defined.")
print("TODO: update supervisor prompt, rebuild graph, test routing order.")

In [ ]:
# ── Exercise 2 Starter — Turn counter guard ───────────────────────────────────

class GuardedAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    turn_count: int  # initialize to 0 when calling app.invoke()


def guarded_supervisor_node(state: GuardedAgentState) -> dict:
    # TODO: read state.get("turn_count", 0) and increment by 1
    # TODO: if new turn_count >= 5, return {"next": "FINISH", "turn_count": new_count}
    # TODO: otherwise proceed with normal supervisor_chain.invoke(...)
    pass


print("Starter: GuardedAgentState with turn_count field.")
print("TODO: implement guarded_supervisor_node, rebuild graph, verify termination.")

---

## What's Next?

You now understand the supervisor/worker pattern end-to-end. Here is where to go deeper:

### More multi-agent patterns
- **Example 6 — Multi-Agent Graph** ([`../6-multi-agent/`](../6-multi-agent/)): same pattern as a runnable script — good comparison to see what the notebook abstracts away.
- **Example 43 — Supervisor/Worker (script)** ([`../43-supervisor-worker/`](../43-supervisor-worker/)): three specialist workers in a production-style script.

### Add intelligence to retrieval
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): route queries to different retrieval strategies based on complexity — combine with the researcher worker here.
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): agent decides when to retrieve, what to retrieve, and whether to iterate.

### Go to production
- **LangGraph Platform**: deploy graphs with persistent storage and REST APIs — https://langchain-ai.github.io/langgraph/concepts/langgraph_platform/
- **LangSmith**: trace every supervisor decision and worker call in a visual dashboard — https://smith.langchain.com
- **Async execution**: all LangGraph nodes support `ainvoke()` for non-blocking use in web servers.

### Further reading
- Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442
- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation Framework.* https://arxiv.org/abs/2308.08155
- LangGraph multi-agent concepts: https://langchain-ai.github.io/langgraph/concepts/multi_agent/
- LangGraph `Send` API (fan-out / map-reduce): https://langchain-ai.github.io/langgraph/how-tos/map-reduce/

---

*Workshop complete. The next step is Example 25 — add adaptive retrieval intelligence to your researcher worker.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions — not the only correct approaches.

### Exercise 1 — Fact-Checker Worker

**Key insight:** Adding a new worker requires three coordinated changes:
1. Extend `SupervisorDecision.next` with the new worker name in the `Literal`
2. Add the node to the graph and wire its edges
3. Update the supervisor prompt to explain *when* to call the new worker

If you do steps 1 and 2 without step 3, the supervisor will never route there — it does not learn worker names from the graph topology; it learns them only from the prompt.

**Expected routing:** `supervisor -> researcher -> supervisor -> writer -> supervisor -> checker -> supervisor -> FINISH`

In [ ]:
# Answer Key — Exercise 1: fact-checker worker

class SupervisorDecisionFull(BaseModel):
    next: Literal["researcher", "writer", "checker", "FINISH"]


SUPERVISOR_PROMPT_FULL = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a supervisor coordinating three specialist agents:
- researcher: searches the web for current facts
- writer:     synthesizes research into a clear answer
- checker:    verifies the writer's claims against the researcher's findings

Rules:
1. Always call researcher first.
2. Call writer after researcher.
3. Call checker after writer.
4. Reply FINISH after checker confirms the answer.
5. Never call the same agent twice in a row.""",
        ),
        ("human", "Conversation so far:\n{history}\n\nWho should act next?"),
    ]
)

supervisor_chain_full = SUPERVISOR_PROMPT_FULL | llm.with_structured_output(
    SupervisorDecisionFull
)


def supervisor_node_full(state: AgentState) -> dict:
    history = "\n".join(
        f"{(getattr(m, 'name', None) or m.type).upper()}: {m.content[:300]}"
        for m in state["messages"]
    )
    decision = supervisor_chain_full.invoke({"history": history})
    return {"next": decision.next}


def route_full(state: AgentState) -> str:
    nxt = state.get("next", "FINISH")
    return "__end__" if nxt == "FINISH" else nxt


graph_full = StateGraph(AgentState)
graph_full.add_node("supervisor", supervisor_node_full)
graph_full.add_node("researcher", researcher_node)
graph_full.add_node("writer", writer_node)
graph_full.add_node("checker", checker_node)

graph_full.add_edge(START, "supervisor")
graph_full.add_conditional_edges(
    "supervisor",
    route_full,
    {"researcher": "researcher", "writer": "writer", "checker": "checker", "__end__": END},
)
graph_full.add_edge("researcher", "supervisor")
graph_full.add_edge("writer", "supervisor")
graph_full.add_edge("checker", "supervisor")

app_full = graph_full.compile()
print("Graph with fact-checker compiled: researcher + writer + checker.")

### Exercise 2 — Turn Counter Guard

**Why this matters in production:** A supervisor with a subtle prompt bug can route indefinitely, consuming tokens and never terminating. The turn counter is a hard circuit-breaker that costs nothing on happy paths.

**Tip:** Log when the guard fires — in production, this is a signal that your supervisor prompt needs tightening. The guard is a safety net, not a design target.

In [ ]:
# Answer Key — Exercise 2: turn counter

class GuardedState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    turn_count: int


HARD_LIMIT = 5


def guarded_supervisor(state: GuardedState) -> dict:
    turn = state.get("turn_count", 0) + 1
    if turn >= HARD_LIMIT:
        print(f"  [GUARD] Forcing FINISH at turn {turn}")
        return {"next": "FINISH", "turn_count": turn}
    history = "\n".join(
        f"{(getattr(m, 'name', None) or m.type).upper()}: {m.content[:300]}"
        for m in state["messages"]
    )
    decision = supervisor_chain.invoke({"history": history})
    print(f"  [supervisor turn {turn}] -> {decision.next}")
    return {"next": decision.next, "turn_count": turn}


def guarded_route(state: GuardedState) -> str:
    nxt = state.get("next", "FINISH")
    return "__end__" if nxt == "FINISH" else nxt


guarded_graph = StateGraph(GuardedState)
guarded_graph.add_node("supervisor", guarded_supervisor)
guarded_graph.add_node("researcher", researcher_node)
guarded_graph.add_node("writer", writer_node)
guarded_graph.add_edge(START, "supervisor")
guarded_graph.add_conditional_edges(
    "supervisor",
    guarded_route,
    {"researcher": "researcher", "writer": "writer", "__end__": END},
)
guarded_graph.add_edge("researcher", "supervisor")
guarded_graph.add_edge("writer", "supervisor")
guarded_app = guarded_graph.compile()

print(f"Guarded graph compiled. Hard limit: {HARD_LIMIT} supervisor turns.")
print()

# Smoke test
test_result = guarded_app.invoke(
    {
        "messages": [HumanMessage(content="What is the capital of France?")],
        "next": "",
        "turn_count": 0,
    }
)
print(f"Completed in {test_result.get('turn_count', 0)} supervisor turn(s).")
print(f"Final answer: {test_result['messages'][-1].content[:120]}...")